## Segunda parte do tratamento da base de dados do Bolsa Família

#### 1 - Abrir o arquivo bolsafamilia tratado em parquet para continuar o tratamento e treinamento:

In [12]:
import polars as pl
import plotly.express as px

bolsa2 = "bolsapl_tratado.parquet"

df_lazy = pl.scan_parquet(bolsa2)

df_lazy

### 2 - Verifica a quantidade de beneficiários (usuários) únicos existentes na base:

In [2]:
df_lazy.select(
    pl.col("nis").n_unique().alias("beneficiarios_unicos")
).collect()

beneficiarios_unicos
u32
20486709


### 3 - Verificar o esquema de colunas da base, de quais tipos são:

In [3]:
df_lazy.collect_schema()

Schema([('mes_comp', Int64),
        ('mes_ref', Int64),
        ('uf', String),
        ('cod_municipio', Int64),
        ('municipio', String),
        ('nis', String),
        ('favorecido', String),
        ('parcela', Float64)])

### 4 - Realizar as estatísticas descritivas da coluna 'parcela', usando o modo lazy:

In [4]:
df_lazy.select([
    pl.col("parcela").mean().alias("media"),
    pl.col("parcela").median().alias("mediana"),
    pl.col("parcela").min().alias("minimo"),
    pl.col("parcela").max().alias("maximo"),
    pl.col("parcela").std().alias("desvio_padrao"),
    pl.col("parcela").sum().alias("total_pago")
]).collect()

media,mediana,minimo,maximo,desvio_padrao,total_pago
f64,f64,f64,f64,f64,f64
670.091255,650.0,25.0,3938.0,189.928508,2.7347e10


### 5 - Criando faixas de valor para as parcelas e verificar quantos beneficiários se encaixam em cada uma delas:

In [11]:
df_faixas = (
    df_lazy
    .with_columns(
        pl.when(pl.col("parcela") < 200).then(pl.lit("0-199"))
        .when(pl.col("parcela") < 400).then(pl.lit("200-399"))
        .when(pl.col("parcela") < 600).then(pl.lit("400-599"))
        .when(pl.col("parcela") < 800).then(pl.lit("600-799"))
        .when(pl.col("parcela") < 1000).then(pl.lit("800-999"))
        .otherwise(pl.lit("1000+"))
        .alias("faixa_valor")
    )
    .group_by("faixa_valor")
    .agg([
        pl.col("parcela").count().alias("quantidade"),
        pl.col("parcela").mean().alias("media_faixa")
    ])
)

df_faixas = df_faixas.collect()

ordem_correta = ["0-199", "200-399", "400-599", "600-799", "800-999", "1000+"]
ordem_dict = {v: i for i, v in enumerate(ordem_correta)}

df_faixas = (
    df_faixas
    .with_columns(
        pl.col("faixa_valor")
        .replace(ordem_dict)
        .alias("ordem")
    )
    .sort("ordem")
    .drop("ordem")
)

df_faixas

faixa_valor,quantidade,media_faixa
str,u32,f64
"""0-199""",845,46.775148
"""200-399""",4156101,330.172578
"""400-599""",1564080,443.385141
"""600-799""",26235366,653.384646
"""800-999""",7015353,850.473527
"""1000+""",1839248,1181.55477


### 6 - Criar gráfico de barras apresentando a quantidade de beneficiários por faixa etária, baseado no código anterior:

In [19]:
fig = px.bar(
    df_faixas.to_pandas(),
    x="quantidade",
    y="faixa_valor",
    orientation="h",
    text="quantidade",
    title="Distribuição de Beneficiários por Faixa de Valor do Bolsa Família",
    color="faixa_valor",
    color_discrete_sequence=px.colors.sequential.Tealgrn
)

fig.update_traces(
    texttemplate='%{text:,.0f}', 
    textposition='outside')
fig.update_layout(
    xaxis_title="Número de Beneficiários",
    yaxis_title="Faixa de Valor",
    showlegend=False,
    plot_bgcolor='rgba(0,0,0,0)',
    xaxis=dict(showgrid=True),
    yaxis=dict(categoryorder='array', categoryarray=df_faixas["faixa_valor"].to_list())
)

fig.show()

fig.write_html("grafico_beneficiarios_por_faixa_etaria.html")